# Werewolf Transformer — low-thousands pilot (iPad / Google Colab)

これは smoke run の次段階です。**新しい `pilot-002`** として、

1. 1024村の general self-play bootstrap
2. empirical population iteration を2世代
3. 各世代で Village / Werewolf / Fox の oracle を256村ずつ

まで実行します。

学習状態はすべて Google Drive に保存します。Colab が切れても、このノートブックを再び上から実行すれば bootstrap / population の両方が自動 resume します。

### あなたが行うこと
1. Colab でこのノートブックを開く
2. **ランタイム → ランタイムのタイプを変更 → GPU** を選ぶ
3. 上から順にセルを実行する
4. 最後に表示される `PILOT-002 REPORT` を ChatGPT に送る

`pilot-001` の小規模検証データは変更しません。報酬・ルール・action mask・観測設計も変更しません。


In [ ]:
# 1) Google Drive を接続
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2) 最新 main を取得して依存関係をインストール
%cd /content
!rm -rf Are-you-werewolf
!git clone --depth 1 https://github.com/dolphin23-jp/Are-you-werewolf.git
%cd /content/Are-you-werewolf/backend
!python -m pip install -q -e ".[rl,transformer]"


In [ ]:
# 3) GPU と保存先を確認
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU が有効ではありません。Colab の「ランタイム → ランタイムのタイプを変更」で GPU を選んでから、最初から実行してください。'
    )

RUN_ROOT = Path('/content/drive/MyDrive/werewolf-training/pilot-002')
BOOTSTRAP = RUN_ROOT / 'bootstrap'
POOL = RUN_ROOT / 'pool'
POPULATION = RUN_ROOT / 'population'

BOOTSTRAP.mkdir(parents=True, exist_ok=True)
POOL.mkdir(parents=True, exist_ok=True)
POPULATION.mkdir(parents=True, exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('保存先:', RUN_ROOT)
print('pilot-001 とは別保存なので既存結果は変更しません。')


In [ ]:
# 4) 1024村 general self-play bootstrap
#    64村ごとに checkpoint / run-state を Drive に保存し、中断後は自動 resume します。
import subprocess

model = BOOTSTRAP / 'model.npz'
run_state = BOOTSTRAP / 'run.npz'
metrics = BOOTSTRAP / 'metrics.jsonl'

common = [
    'python', 'scripts/train_self_play_torch.py',
    '--episodes', '1024',
    '--pool-dir', str(POOL),
    '--output', str(model),
    '--run-state', str(run_state),
    '--metrics-jsonl', str(metrics),
    '--device', 'auto',
]

if run_state.exists():
    cmd = common + ['--resume']
    print('既存の bootstrap run-state を検出: 1024村まで resume します。')
else:
    if model.exists() or metrics.exists():
        raise RuntimeError(
            'bootstrap/run.npz が無いのに model または metrics が残っています。'
            '不完全な手動変更の可能性があるため、ChatGPT にこの表示を送ってください。'
        )
    cmd = common + [
        '--batch-size', '64',
        '--parallel-games', '16',
        '--inference-batch-size', '64',
        '--seed', '2001',
    ]
    print('新しい1024村 bootstrap を開始します。')

subprocess.run(cmd, check=True)
print('\nBOOTSTRAP TARGET REACHED')


In [ ]:
# 5) bootstrap の状態を確認
import json

rows = [
    json.loads(line)
    for line in metrics.read_text().splitlines()
    if line.strip()
]
if not rows:
    raise RuntimeError('bootstrap metrics がありません。直前セルの出力を確認してください。')

last = rows[-1]
print('bootstrap completed episodes:', last['completed_episodes'])
print('bootstrap batches:', len(rows))
print('last rollout eps/s:', last['rollout_episodes_per_second'])
print('last KL:', last['ppo']['mean_approx_kl'])
print('last entropy:', last['ppo']['mean_path_entropy'])
print('last value EV:', last['ppo']['rollout_value_explained_variance'])

if last['completed_episodes'] < 1024:
    raise RuntimeError('1024村に未到達です。セル4をもう一度実行して resume してください。')


In [ ]:
# 6) population iteration を2世代実行
#    各 iteration:
#      - recent policy 3本/陣営
#      - 27 profile を基本5戦ずつ
#      - uncertainty allocation +32戦
#      - Village/Werewolf/Fox oracle 各256村
#    outer state / oracle state は Drive に逐次保存されます。
import subprocess
from pathlib import Path

population_state = POPULATION / 'population.run.json'
events_log = POPULATION / 'events.log'

common = [
    'python', 'scripts/run_population_iterations_torch.py',
    '--pool-dir', str(POOL),
    '--run-dir', str(POPULATION),
    '--iterations', '2',
    '--device', 'auto',
]

if population_state.exists():
    cmd = common + ['--resume']
    print('既存 population state を検出: iteration 2 完了まで resume します。')
else:
    cmd = common + [
        '--recent-policies', '3',
        '--games-per-profile', '5',
        '--extra-games', '32',
        '--oracle-episodes', '256',
        '--oracle-batch-size', '32',
        '--parallel-games', '16',
        '--inference-batch-size', '64',
        '--evaluation-seed', '2101',
        '--oracle-seed', '2201',
        '--opponent-seed', '2301',
    ]
    print('population research iteration 1 → 2 を開始します。')

events_log.parent.mkdir(parents=True, exist_ok=True)
with events_log.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

print('\nPOPULATION TARGET REACHED')


In [ ]:
# 7) ChatGPT に送る PILOT-002 REPORT を表示
import json
from pathlib import Path

print('\n===== PILOT-002 REPORT =====')

# Bootstrap: last 5 batches
bootstrap_rows = [
    json.loads(line)
    for line in (BOOTSTRAP / 'metrics.jsonl').read_text().splitlines()
    if line.strip()
]
print('\n===== BOOTSTRAP LAST 5 =====')
for row in bootstrap_rows[-5:]:
    compact = {
        'batch': row['batch'],
        'completed_episodes': row['completed_episodes'],
        'wins': row['wins'],
        'mean_days': row['mean_days'],
        'mean_decisions': row['mean_decisions'],
        'rollout_episodes_per_second': row['rollout_episodes_per_second'],
        'inference': row['inference'],
        'ppo': row['ppo'],
    }
    print(json.dumps(compact, ensure_ascii=False))
print('===== END BOOTSTRAP LAST 5 =====')

# Population summaries: iteration 1 and 2
for iteration in (1, 2):
    summary_path = POPULATION / f'iteration-{iteration:04d}' / 'summary.json'
    print(f'\n===== POPULATION SUMMARY {iteration} =====')
    if summary_path.exists():
        print(json.dumps(json.loads(summary_path.read_text()), ensure_ascii=False, indent=2))
    else:
        print(f'MISSING: {summary_path}')
    print(f'===== END POPULATION SUMMARY {iteration} =====')

# Last learning events
events_log = POPULATION / 'events.log'
print('\n===== LEARNING EVENTS (LAST 50) =====')
if events_log.exists():
    lines = [line for line in events_log.read_text().splitlines() if line.strip()]
    for line in lines[-50:]:
        print(line)
else:
    print('MISSING events.log')
print('===== END LEARNING EVENTS =====')

print('\n===== END PILOT-002 REPORT =====')
print('\nこの PILOT-002 REPORT 全体を ChatGPT に貼り付けてください。')


## 途中で Colab が切れた場合

同じノートブックを**上から再実行**してください。

- セル4は `bootstrap/run.npz` があれば `--resume`
- セル6は `population/population.run.json` があれば `--resume`

を自動で選びます。

1024村 bootstrap が終わっていればセル4は実質何も追加せず、そのまま population へ進めます。
population iteration 2まで完了していればセル6も追加学習せず終了します。

**手動で Drive 内の `pilot-002` のファイルを移動・改名・削除しないでください。**
